# Compiler Optimization

(This content was generate by Claude Sonnet 5.0 based on prior lectures from Randal Burns and edited by Randal Burns.)

## What is a compiler actually doing?

When you write `x = a + b * c`, you're describing *what* to compute, not *how*
a processor should compute it. The compiler's job is to turn that description
into instructions a real CPU can execute — and along the way, it's allowed to
pick a different sequence of instructions than the obvious, line-by-line
translation, as long as the program still produces the same observable
result: same output, same return value, same side effects (files written,
things printed).

**That's the whole idea of "optimization."** It isn't one thing the compiler
does — it's a long pipeline of small, independent rewrites, each looking for
one pattern:

* "this value is computed but never used" &rarr; delete it
* "this variable never has its address taken" &rarr; keep it in a register
  instead of memory
* "these four loop iterations don't depend on each other" &rarr; pack them
  into one vector instruction
* "this branch's condition is always true at compile time" &rarr; delete the
  branch and the dead side

Every one of these rewrites has to be *provably* safe. If the compiler can't
prove two versions of the code behave identically — because a pointer might
alias, because floating-point addition isn't associative, because a function
call might have side effects the compiler can't see into — it leaves your
code alone, even if a human could look at it and know the rewrite is fine.
The compiler is not a genius partner reading your intent; it's a very
literal-minded prover that only acts on what it can verify.

## Why "levels" exist at all

If the compiler can prove a rewrite is safe, why wouldn't it always do it?
Two reasons:

* **Compile time.** Some analyses are cheap (does this value get used?);
  others are expensive (can I prove these two array accesses never
  overlap, across the whole function?). Running every possible analysis on
  every build would make compiling agonizingly slow for no benefit while
  you're still writing and debugging the code.
* **Debuggability.** At no optimization, one line of source maps to a
  predictable, self-contained chunk of instructions, so a debugger can step
  "line by line" and show you variables sitting exactly where you'd expect.
  Once the compiler starts reordering, merging, and deleting code, that
  correspondence breaks down — you can still debug optimized code, but it's
  much harder to reason about.

An **optimization level** (`-O0`, `-O1`, `-O2`, `-O3`, ...) is just a preset
list of which rewrite passes to run and how aggressively to run them. It's a
knob on effort and risk, not a single switch — turning on `-O2` really means
running one specific bundle of a few dozen independent passes.

## The rule underneath all of it: don't change what's observable

The formal name for this is the **as-if rule**: the compiler may transform
your program into anything it likes, as long as the transformed program
behaves *as if* it had run your original code.

[pipeline.dce.md](../examples/pipeline/pipeline.dce.md) is the clearest
demonstration in this course. A dead store (a write nothing ever reads), a
branch whose condition is a compile-time constant, and a call to a function
whose result is discarded all get deleted entirely once optimization is on —
not sped up, *removed*, because removing them can't change what the program
observably does. The write-up's numbers: ~22 ms &rarr; 0 ms (dead store) and
~28 ms &rarr; 0 ms (dead call) once optimization kicks in — the work simply
never gets generated.

The exception is `-O0`, which is instructive precisely because it *is* the
literal, unoptimized translation: the dead-branch "free win" disappears
completely (no optimization pass exists yet to notice the branch is dead),
and the dead call balloons to ~5.8 seconds, because the unused function it
calls is itself compiled with no optimization either. `-O0` isn't "a little
bit of optimization" — it's none at all.

## The levels, assuming `clang`

| Level | What it turns on | What you get | Watch out for | See it in this course |
|---|---|---|---|---|
| `-O0` | Nothing. Values reload from the stack between statements; no inlining, no dead-code removal, no vectorization. | Fast compiles, and a debugger can trust that each source line maps to its own instructions. | This is a **debugging** setting, not a performance baseline. Hand-optimizations can look actively harmful here — [loop_unrolling.md](../examples/loop_optimizations/loop_unrolling.md) shows manual unrolling running *slower* than the plain scalar loop at `-O0` (0.69x), purely from stack traffic the optimizer would normally remove. |
| `-O1` | Basic, cheap passes: promoting stack variables to registers, constant folding/propagation, dead-store and dead-code elimination, inlining tiny functions, straight-line common-subexpression elimination. No auto-vectorization, no aggressive loop transforms. | The noise from `-O0`'s literal translation is gone, but the compiler hasn't yet found *your* optimization on its own — so hand-written fixes show their real effect. | It's easy to mistake `-O1` for "close to `-O2`." It isn't — no vectorizer runs here at all. | [pipeline.md](../examples/pipeline/pipeline.md): breaking a dependency chain into 4 independent accumulators is 3.59x faster at `-O1`, "the cleanest run." |
| `-O2` | Adds the loop and SLP auto-vectorizers, more aggressive inlining budgets, global value numbering, loop unrolling/rotation, strength reduction. clang's recommended default for release builds. | The compiler starts finding several of the tricks you'd hand-apply, on its own. | A hand-optimization that won at `-O1` can lose at `-O2` once the compiler automates the same idea — see below. | [loop_unrolling.md](../examples/loop_optimizations/loop_unrolling.md): the manual 4x unroll (0.83x) now *loses* to the plain scalar loop, because the compiler's own unrolling already beats it. |
| `-O3` | The same pass pipeline as `-O2`, run with more aggressive thresholds — bigger inlining budgets, more loop unrolling and vectorization interleaving. | Occasionally a further win over `-O2`. | Often *not* a further win — bigger code can mean more instruction-cache pressure, and compile time grows for a result that's frequently identical to `-O2`. | [pipeline.cse.md](../examples/pipeline/pipeline.cse.md): the loop-index CSE case's headline 17x speedup is specific to `-O2`/`-O3` — it's a more modest ~1.4-2.1x at `-O0`/`-O1`. |
| `-Ofast` | `-O3`, plus relaxed floating-point semantics (`-ffast-math` and friends): the compiler is allowed to treat FP addition/multiplication as associative and assume no `NaN`/`Inf`. | Real speedups on FP-heavy code that `-O3` alone can't touch. | This one is not "as-if" safe — it changes results. [loop_fusion.md](../examples/loop_optimizations/loop_fusion.md)'s mean/variance reduction won't auto-vectorize without it, precisely *because* floating-point addition isn't associative and the compiler refuses to reorder it on your behalf otherwise. Not used elsewhere in this course. |
| `-Os` / `-Oz` | Roughly `-O2` with size-increasing passes (aggressive inlining, unrolling) dialed back or removed; `-Oz` goes further. | Smaller binaries, for size-constrained targets (embedded, mobile). | Not a performance level, and not used in this course — mentioned here so the names aren't a mystery if you see them elsewhere. |

## Why we rely on `-O1` for many examples?

`-O0` hides real effects behind stack-traffic noise, and `-O2`/`-O3` tend to
*discover the same optimization the example is trying to teach you*, which
erases the comparison you're trying to see. `-O1` is the level where the
compiler has cleaned up the accidental slowness of `-O0` but hasn't yet
automated the specific trick a given example is about — so the effect shows
up honestly, as a timing difference you can measure.

Many of our examples run well at `-02` also if you turn off the auto-vectorizer.

[pipeline/README.md](../examples/pipeline/README.md) makes this explicit for
every example in that directory with a "Verified optimization levels" table,
confirming which levels actually show the effect as a timing comparison and
which ones the compiler has already solved for you.